In [37]:
import requests 
import pandas 
import io

ModuleNotFoundError: No module named 'PIL'

In [84]:
from datetime import datetime


url = 'https://pixel.wp.com/c.gif?s=2&u=https%3A%2F%2Fs3-eu-central-1.amazonaws.com%2Ftso-cy%2FMMS-dry-run%2FFCST%2FISP_FCST%2Fexcel%2F2022%2F11%2FISP_FCST_20221130.xlsx&r=noopener&b=173746534&p=5314&rand=0.32547418589445054'

url2 = "https://s3-eu-central-1.amazonaws.com/tso-cy/MMS-dry-run/FCST/ISP_FCST/excel/2022/12/ISP_FCST_20221028.xlsx"

def getTsocMMS(date: datetime):
    url = f'https://s3-eu-central-1.amazonaws.com/tso-cy/MMS-dry-run/FCST/ISP_FCST/excel/{date.strftime("%Y")}/{date.strftime("%m")}/ISP_FCST_{date.strftime("%Y%m%d")}.xlsx'



    columns = ['Timestamp', 'TotalDemandForecastAtReferencePoint', 'TotalFBproductionForecast', 'TotalWindProductionForecast', 'TotalAPEProductionForecast', 'TotalUprisingESSForecast', 'TotalUprisingaEASForecast', 'TotalUprisingchEASForecast', 'TotalDownwardsESSForecast', 'TotalDownwardsaEASForecast', 'TotalDownwardschEASForecast', 'MinimumRequirementYEMP', 'InstalledFBPower', 'InstalledWindPower']

    r = requests.get(url)
    print(r.status_code)
    df = pandas.read_excel(io.BytesIO(r.content),  sheet_name = 'Προβλέψεις')

    df = df.drop(["Περίοδος",f"Πρόβλεψη ΔΟΠ για τις {date.strftime('%d/%m/%Y')}",'Unnamed: 3'],axis=1)
    df.columns = columns
    return df



date = datetime(2022,12,11)
data = getTsocMMS(date)
data.iloc[:,0]

200


0    2022-12-11 00:00:00
1    2022-12-11 00:30:00
2    2022-12-11 01:00:00
3    2022-12-11 01:30:00
4    2022-12-11 02:00:00
5    2022-12-11 02:30:00
6    2022-12-11 03:00:00
7    2022-12-11 03:30:00
8    2022-12-11 04:00:00
9    2022-12-11 04:30:00
10   2022-12-11 05:00:00
11   2022-12-11 05:30:00
12   2022-12-11 06:00:00
13   2022-12-11 06:30:00
14   2022-12-11 07:00:00
15   2022-12-11 07:30:00
16   2022-12-11 08:00:00
17   2022-12-11 08:30:00
18   2022-12-11 09:00:00
19   2022-12-11 09:30:00
20   2022-12-11 10:00:00
21   2022-12-11 10:30:00
22   2022-12-11 11:00:00
23   2022-12-11 11:30:00
24   2022-12-11 12:00:00
25   2022-12-11 12:30:00
26   2022-12-11 13:00:00
27   2022-12-11 13:30:00
28   2022-12-11 14:00:00
29   2022-12-11 14:30:00
30   2022-12-11 15:00:00
31   2022-12-11 15:30:00
32   2022-12-11 16:00:00
33   2022-12-11 16:30:00
34   2022-12-11 17:00:00
35   2022-12-11 17:30:00
36   2022-12-11 18:00:00
37   2022-12-11 18:30:00
38   2022-12-11 19:00:00
39   2022-12-11 19:30:00


In [89]:
from typing import Optional
from pydantic import BaseModel
import pandas as pd


class TsocForecastData(BaseModel):
    Timestamp : Optional[datetime] 
    TotalDemandForecast : Optional[float] = None
    TotalSolarProductionForecast : Optional[float]
    TotalWindProductionForecast: Optional[float]  
    FCRUpForecast : Optional[float]
    AFRRUpForecast : Optional[float]
    MFRRUpForecast : Optional[float]
    FCRDownForecast : Optional[float]
    AFRRDownForecast : Optional[float]
    MFRRDownForecast : Optional[float]
    ThermalCommisioningForecast : Optional[float]
    InstalledSolarPower : Optional[float]
    InstalledWindPower : Optional[float]


def tsocForecastUrl(date: datetime):
    return f'https://s3-eu-central-1.amazonaws.com/tso-cy/MMS-dry-run/FCST/ISP_FCST/excel/{date.strftime("%Y")}/{date.strftime("%m")}/ISP_FCST_{date.strftime("%Y%m%d")}.xlsx'

tsocForecastUrl(date)

'https://s3-eu-central-1.amazonaws.com/tso-cy/MMS-dry-run/FCST/ISP_FCST/excel/2022/12/ISP_FCST_20221211.xlsx'

In [106]:
import pandas as pd 

def getTsocMMS(date: datetime):
    
    url = tsocForecastUrl(date)


    columns = ['Timestamp',
               'TotalDemandForecast',
               'TotalSolarProductionForecast',
               'TotalWindProductionForecast',
               'FCRUpForecast',
               'AFRRUpForecast',
               'MFRRUpForecast',
               'FCRDownForecast',
               'AFRRDownForecast',
               'MFRRDownForecast', 
               'ThermalCommisioningForecast',
               'InstalledSolarPower',
               'InstalledWindPower']

    r = requests.get(url)
    print(r.status_code)
    df = pd.read_excel(io.BytesIO(r.content),  sheet_name = 'Προβλέψεις')

    #drop unwanted columns
    df = df.drop(["Περίοδος", "Πρόβλεψη ολικής παραγωγής ΑΠΕ \n(MW)"
                ,f"Πρόβλεψη ΔΟΠ για τις {date.strftime('%d/%m/%Y')}",'Unnamed: 3'],axis=1)
    #add column names
    df.columns = columns

    
    #final_data = df.apply(lambda row: TsocForecastData(**row.to_dict()), axis=1)

    return df['Timestamp']

date = datetime(2022,12,11)
date = getTsocMMS(date)


200


In [107]:
def convert_to_utc(timestamp):
    if isinstance(timestamp, pd.Timestamp):
        timestamp_converted = timestamp.tz_localize('Europe/Athens')
        return timestamp_converted.tz_convert('UTC')
    else:
        raise ValueError("Input must be a Pandas timestamp.")
    



date = date.apply(convert_to_utc)
date

0    2022-12-10 22:00:00+00:00
1    2022-12-10 22:30:00+00:00
2    2022-12-10 23:00:00+00:00
3    2022-12-10 23:30:00+00:00
4    2022-12-11 00:00:00+00:00
5    2022-12-11 00:30:00+00:00
6    2022-12-11 01:00:00+00:00
7    2022-12-11 01:30:00+00:00
8    2022-12-11 02:00:00+00:00
9    2022-12-11 02:30:00+00:00
10   2022-12-11 03:00:00+00:00
11   2022-12-11 03:30:00+00:00
12   2022-12-11 04:00:00+00:00
13   2022-12-11 04:30:00+00:00
14   2022-12-11 05:00:00+00:00
15   2022-12-11 05:30:00+00:00
16   2022-12-11 06:00:00+00:00
17   2022-12-11 06:30:00+00:00
18   2022-12-11 07:00:00+00:00
19   2022-12-11 07:30:00+00:00
20   2022-12-11 08:00:00+00:00
21   2022-12-11 08:30:00+00:00
22   2022-12-11 09:00:00+00:00
23   2022-12-11 09:30:00+00:00
24   2022-12-11 10:00:00+00:00
25   2022-12-11 10:30:00+00:00
26   2022-12-11 11:00:00+00:00
27   2022-12-11 11:30:00+00:00
28   2022-12-11 12:00:00+00:00
29   2022-12-11 12:30:00+00:00
30   2022-12-11 13:00:00+00:00
31   2022-12-11 13:30:00+00:00
32   202

In [3]:
L = list(range(1,51))
print(L)
for i in L:
    for j in range(1,3):
        print(2*(i - 1) + j)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100


In [73]:
hours_of_day = ["00:00", "00:30", "01:00", "01:30", "02:00", "02:30", "02:00", "02:30", "03:00", "03:30", 
                "04:00", "04:30", "05:00", "05:30", "06:00", "06:30", "07:00", "07:30", 
                "08:00", "08:30", "09:00", "09:30", "10:00", "10:30", "11:00", "11:30", 
                "12:00", "12:30", "13:00", "13:30", "14:00", "14:30", "15:00", "15:30", 
                "16:00", "16:30", "17:00", "17:30", "18:00", "18:30", "19:00", "19:30", 
                "20:00", "20:30", "21:00", "21:30", "22:00", "22:30", "23:00", "23:30"]
#print(len(hours_of_day))


dst_flag = False
keep_hours = []

for hour in hours_of_day:

    hour_data = hour.split(':') 
    print(int(hour_data[0])  * 2 + int(hour_data[1]) // 30 + 1 )
    #print((int(hour_data[0]) + 1) * 2 - (0 if hour_data[1] == '30' else 1))
    #print(int(hour_data[0])  * 2 + int(hour_data[1]) // 30 + 1 )

# dispatch = []
# for i in keep_hours:
#     for j in range(1,3):
#         print(2*(i - 1) + j)

#print(keep_hours)


1
2
3
4
5
6
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48


In [74]:
from datetime import datetime
import pytz 

s = datetime(2023, 10, 29, 0, 30)
timezone = pytz.timezone('Europe/Athens')
loc_date = timezone.localize(s)
utc_date = pytz.utc.normalize(loc_date).astimezone(pytz.utc)
loc_date2 = loc_date.astimezone(pytz.utc)
print(loc_date2)
print(utc_date)
print(s.date())

2023-10-28 21:30:00+00:00
2023-10-28 21:30:00+00:00
2023-10-29


In [66]:
s = datetime(2023, 10, 29, 0, 0)
datetime.strptime('2023-10-29', '%Y-%m-%d')
#datetime(2023, 10, 29, 0, 0).strftime('%Y-%m-%d')

datetime.datetime(2023, 10, 29, 0, 0)

In [71]:
s1 =  [{'DispatchDay': 'date', 'DispatchPeriod': 1, 'EntityName': 'AG_DIMITRIOS1', 'ActivatedBalancingEnergyUp': 0, 'ActivatedBalancingEnergyDown': 0, 'Version': "date2"}
        , {'DispatchDay': 'date', 'DispatchPeriod': 1, 'EntityName': 'AG_DIMITRIOS1', 'ActivatedBalancingEnergyUp': 0, 'ActivatedBalancingEnergyDown': 0, 'Version': "date2"}]

In [79]:
from datetime import timedelta
datetime.strptime('2023-10-29 02:00:00.1', '%Y-%m-%d %H:%M:%S') - timedelta(milliseconds=10)

ValueError: unconverted data remains: .1

In [5]:
from datetime import datetime
string = "2023-10-29 02:30:00.1"

datetime.strptime(string, '%Y-%m-%d %H:%M:%S.%f')

datetime.datetime(2023, 10, 29, 2, 30, 0, 100000)

In [8]:
from dateutil import parser

def differentiate_strings(input_str):
    try:
        # Try to parse the string as a datetime
        datetime_obj = parser.parse(input_str)
        return datetime_obj
    except ValueError:
        # If parsing fails, assume it's not a datetime
        return input_str

# Example usage:
string1 = '2023-10-29 02:30:00.1'
string2 = 'Non Dispatchable Load'

print(differentiate_strings(string1))
differentiate_strings(string2)

2023-10-29 02:30:00.100000


'Non Dispatchable Load'

In [10]:
def checkDateToString(date : str):
    try:
        if parser.parse(date):
            return True
    except ValueError:
        return False
    
print(checkDateToString(string2))
checkDateToString(string1)

False


True

In [13]:
from datetime import datetime   
datetime.strptime('29/10/2023','%d/%m/%Y')

datetime.datetime(2023, 10, 29, 0, 0)

In [68]:
from datetime import datetime, timedelta
import pytz



def getUtcTimestamp(date: datetime):
    #str_to_date =  datetime.strptime(date,'%Y-%m-%d %H:%M:%S')
    cet_tz = pytz.timezone('Europe/Berlin')
    utc_tz = pytz.timezone('UTC')

    cet_day = cet_tz.localize(date)
    last_date = cet_day
    utc_day = last_date.astimezone(utc_tz)
    #utc_day += timedelta(hours)

    return utc_day #+ timedelta(hours = 2)

date = datetime.strptime("29-10-2023 02:00",'%d-%m-%Y %H:%M')

time = getUtcTimestamp(date)
time



#time.total_seconds()

datetime.datetime(2023, 10, 28, 22, 0, tzinfo=<UTC>)

In [84]:
def getTimeDiffernce(date : datetime):
    
    # Convert the UTC datetime to CET
    cet_timezone = pytz.timezone('Europe/Berlin')
    cet_datetime = date.astimezone(cet_timezone)

    # Get the date in CET
    cet_date = cet_datetime.date()

    # Convert the CET date back to a datetime object (midnight)
    cet_datetime_midnight = datetime.combine(cet_date, datetime.min.time())

    # Convert the CET datetime back to UTC
    cet_datetime_midnight = cet_timezone.localize(cet_datetime_midnight)

    #normalize
    utc_date = pytz.utc.normalize(cet_datetime_midnight)

    hour_diff = date-utc_date

    return cet_datetime_midnight, int(abs(hour_diff.total_seconds()) // 3600) + 1

date = datetime.strptime('2023-10-23 00:00:00','%Y-%m-%d %H:%M:%S')
date = date.astimezone(pytz.timezone('Europe/Berlin'))
time = getTimeDiffernce(date)[1]
time


1

In [36]:
import pandas as pd
d = pd.Timestamp('2023-10-23 00:00:00').to_pydatetime()
r = d.astimezone(pytz.timezone('Europe/Berlin'))
r

datetime.datetime(2023, 10, 22, 23, 0, tzinfo=<DstTzInfo 'Europe/Berlin' CEST+2:00:00 DST>)

In [72]:
# Sample DataFrame with a date column
df = pd.DataFrame({'date_column': ['2023-03-26 00:00:00', '2023-10-29 00:00:00', '2023-10-29 03:30:00']})

# Convert the 'date_column' to datetime objects
df['date_column'] = pd.to_datetime(df['date_column'])

# Define timezone for CET
cet_timezone = pytz.timezone('CET')

# Define timezone for UTC
utc_timezone = pytz.utc

# Convert CET to UTC
df['date_column_utc'] = df['date_column'].dt.tz_localize(cet_timezone).dt.tz_convert(utc_timezone)

df['time_difference'] = df['date_column_utc'] - df['date_column_utc'].shift(1)

# Print the DataFrame
print(df)

          date_column           date_column_utc   time_difference
0 2023-03-26 00:00:00 2023-03-25 23:00:00+00:00               NaT
1 2023-10-29 00:00:00 2023-10-28 22:00:00+00:00 216 days 23:00:00
2 2023-10-29 03:30:00 2023-10-29 02:30:00+00:00   0 days 04:30:00


In [87]:
def getTimeDifferncePandas(date : pd.Timestamp):
    
    cet_time = date.tz_localize(pytz.timezone('Europe/Berlin'), ambiguous = True)

    # Convert the UTC datetime to CET
    cet_timezone = pytz.timezone('Europe/Berlin')
    #cet_datetime = date.astimezone(cet_timezone)

    # Get the date in CET
    cet_date = cet_time.date()

    # Convert the CET date back to a datetime object (midnight)
    cet_datetime_midnight = datetime.combine(cet_date, datetime.min.time())

    # Convert the CET datetime back to UTC
    cet_datetime_midnight = cet_timezone.localize(cet_datetime_midnight)

    #normalize
    utc_date = pytz.utc.normalize(cet_datetime_midnight)

    cet_date = pytz.utc.normalize(cet_time)

    hour_diff = cet_date-utc_date

    return int(abs(hour_diff.total_seconds()) // 3600) + 1

date = pd.Timestamp('2023-10-29 23:00:00')
utc_date1 = getTimeDifferncePandas(date)
utc_date1

25

In [96]:
def getTimeDiffernce(date : datetime):
    
    cet_time = date.astimezone(pytz.timezone('Europe/Berlin'))

    # Convert the UTC datetime to CET
    cet_timezone = pytz.timezone('Europe/Berlin')
    #cet_datetime = date.astimezone(cet_timezone)

    # Get the date in CET
    cet_date = cet_time.date()

    # Convert the CET date back to a datetime object (midnight)
    cet_datetime_midnight = datetime.combine(cet_date, datetime.min.time())

    # Convert the CET datetime back to UTC
    cet_datetime_midnight = cet_timezone.localize(cet_datetime_midnight)

    #normalize
    utc_date = pytz.utc.normalize(cet_datetime_midnight)

    cet_date = pytz.utc.normalize(cet_time)

    hour_diff = cet_date-utc_date

    return int(abs(hour_diff.total_seconds()) // 3600) + 1

date = datetime(2023,10,30,0,0)
date2 = datetime(2023, 3, 27, 0, 0)
getTimeDiffernce(date2)

23

In [34]:
from datetime import datetime, timedelta
import pytz
def getTimeDiffernce(date : datetime):
    
    # Convert the UTC datetime to CET
    cet_timezone = pytz.timezone('Europe/Berlin')
    cet_datetime = date.astimezone(cet_timezone)

    # Get the date in CET
    cet_date = cet_datetime.date()

    # Convert the CET date back to a datetime object (midnight)
    cet_datetime_midnight = datetime.combine(cet_date, datetime.min.time())

    # Convert the CET datetime back to UTC
    cet_datetime_midnight = cet_timezone.localize(cet_datetime_midnight)

    #normalize
    utc_date = pytz.utc.normalize(cet_datetime_midnight)

    #local_date = cet_timezone.localize(date)

    local_date = cet_timezone.normalize(date)

    hour_diff = local_date-utc_date

    return cet_datetime_midnight, int(abs(hour_diff.total_seconds()) // 3600)  


date = datetime(2023,10,23,0,0)
getTimeDiffernce(date)[1]

ValueError: Naive time - no tzinfo set

In [38]:
import pandas as pd
def getTimeDifferncePandas(date : pd.Timestamp):
    
    cet_time = date.tz_localize(pytz.timezone('Europe/Berlin'), ambiguous = True)

    # Convert the UTC datetime to CET
    cet_timezone = pytz.timezone('Europe/Berlin')
    #cet_datetime = date.astimezone(cet_timezone)

    # Get the date in CET
    cet_date = cet_time.date()

    # Convert the CET date back to a datetime object (midnight)
    cet_datetime_midnight = datetime.combine(cet_date, datetime.min.time())

    # Convert the CET datetime back to UTC
    cet_datetime_midnight = cet_timezone.localize(cet_datetime_midnight)

    #normalize
    utc_date = pytz.utc.normalize(cet_datetime_midnight)

    cet_date = pytz.utc.normalize(cet_time)

    hour_diff = cet_date-utc_date

    return int(abs(hour_diff.total_seconds()) // 3600) + 1



date = datetime(2023,10,23,0,0)
pd_timestamp = pd.Timestamp(date)
getTimeDifferncePandas(pd_timestamp)

1

In [40]:
def checkIntToString(date : str):
    try:
        if int(date):
            return True
    except ValueError:
        return False
    

checkIntToString('waer')



False

In [1]:
from date_helper import parseDateTimeFromArgs
from sources.admie_download  import downloadIspRequirements
from sources.admie import getMandatoryHydroFromFile

def test_assertion(func_1, func_2, dispatch_range, dateFrom, dateTo, market = None, filterFunc = None):

    json = func_1(parseDateTimeFromArgs(dateFrom), parseDateTimeFromArgs(dateTo), market)

    assert json.State == 1


    for row in json.Data:
        results = func_2(row)
        assert max(results.Data.Results, key= lambda x: x['DispatchPeriod'])['DispatchPeriod'] == dispatch_range


dateFrom = '2023-10-29'
dateTo = '2023-10-29'

files  = downloadIspRequirements(parseDateTimeFromArgs(dateFrom), parseDateTimeFromArgs(dateTo))
example = files.Data[0]




#admie.getMandatoryHydroFromFile(file)

https://www.admie.gr/getOperationMarketFilewRange?dateStart=2023-10-29&dateEnd=2023-10-29&FileCategory=ISP1Requirements --- 200
https://www.admie.gr/getOperationMarketFilewRange?dateStart=2023-10-29&dateEnd=2023-10-29&FileCategory=ISP2Requirements --- 200
https://www.admie.gr/getOperationMarketFilewRange?dateStart=2023-10-29&dateEnd=2023-10-29&FileCategory=ISP3Requirements --- 200
https://www.admie.gr/getOperationMarketFilewRange?dateStart=2023-10-29&dateEnd=2023-10-29&FileCategory=ISP4Requirements --- 200


In [4]:

result = getMandatoryHydroFromFile(example)
files



c:\Users\ipapa\Documents\electricity_market_api\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Envelope(State=1, Data=[CustomFile(Id=0, FileName='20231029_ISP1Requirements_03.xlsx', FileDescription='20231029 ISP1 ISPRequirements v3', FileType='ISP1Requirements', Version=3, Url='https://www.admie.gr/sites/default/files/attached-files/type-file/2023/10/20231029_ISP1Requirements_03.xlsx', PublicationDate=datetime.datetime(2023, 10, 28, 13, 35), TargetDateFrom=datetime.datetime(2023, 10, 29, 0, 0), TargetDateTo=datetime.datetime(2023, 10, 29, 0, 0), ByUser=False, Success=True, StatusCode=2, Log=None), CustomFile(Id=0, FileName='20231029_ISP1Requirements_02.xlsx', FileDescription='20231029 ISP1 ISPRequirements v2', FileType='ISP1Requirements', Version=2, Url='https://www.admie.gr/sites/default/files/attached-files/type-file/2023/10/20231029_ISP1Requirements_02.xlsx', PublicationDate=datetime.datetime(2023, 10, 28, 10, 40), TargetDateFrom=datetime.datetime(2023, 10, 29, 0, 0), TargetDateTo=datetime.datetime(2023, 10, 29, 0, 0), ByUser=False, Success=True, StatusCode=2, Log=None), Cust

In [5]:
import pandas as pd 

pd.Timestamp('2023-10-29T03:00:00')

Timestamp('2023-10-29 03:00:00')